In [ ]:
from pathlib import Path

import seaborn as sns
from bonner.plotting import save_figure
from matplotlib import pyplot as plt

from lib.datasets import (
    compute_shared_stimuli,
    filter_by_stimulus,
    split_by_repetition,
    stringer,
)
from lib.spectra import (
    compute_cross_individual_spectra,
    compute_within_individual_spectra,
    plot_spectra,
)
from lib.utilities import JOURNAL_MATPLOTLIBRC

FIGURES_HOME = Path.cwd().parent / "figures"
FIGURES_HOME.mkdir(exist_ok=True, parents=True)

sns.set_theme(context="paper", style="ticks", rc=JOURNAL_MATPLOTLIBRC)

REFERENCE_INDIVIDUAL = 0

In [ ]:
datasets = {
    session: stringer.load_dataset(
        session=session,
        z_score=True,
    )
    for session in range(len(stringer.SESSIONS) - 1)
}

datasets_within = {
    individual: split_by_repetition(
        filter_by_stimulus(
            dataset,
            stimuli=compute_shared_stimuli([dataset], n_repetitions=2),
        ),
        n_repetitions=2,
    )
    for individual, dataset in datasets.items()
}

shared_stimuli = compute_shared_stimuli(datasets.values(), n_repetitions=2)

datasets_cross = {
    individual: split_by_repetition(
        filter_by_stimulus(dataset, stimuli=shared_stimuli),
        n_repetitions=2,
    )
    for individual, dataset in datasets.items()
}

In [ ]:
spectra_within = compute_within_individual_spectra(
    datasets_within,
    n_permutations=5_000,
    stop=2_800,
)
spectra_cross = compute_cross_individual_spectra(
    datasets_cross,
    reference_individual=REFERENCE_INDIVIDUAL,
    n_permutations=5_000,
    stop=2_800,
)

In [ ]:
fig, axes = plt.subplots(
    figsize=(5, 3),
    ncols=2,
    sharex=True,
)

plot_spectra(
    spectra=spectra_within,
    ax=axes[0],
    hue="individual",
    hue_order=list(reversed(range(len(stringer.SESSIONS) - 1))),
    hue_labels=[
        f"{session + 1}" for session in reversed(range(len(stringer.SESSIONS) - 1))
    ],
    palette="crest",
    marker="s",
    hide_insignificant=True,
    null_quantile=0.999,
)

plot_spectra(
    spectra=spectra_cross,
    ax=axes[1],
    hue="individual",
    hue_reference=REFERENCE_INDIVIDUAL,
    hue_order=list(reversed(range(len(stringer.SESSIONS) - 1))),
    hue_labels=[
        f"{session + 1}*" if session == REFERENCE_INDIVIDUAL else f"{session + 1}"
        for session in reversed(range(len(stringer.SESSIONS) - 1))
    ],
    palette="flare",
    marker=None,
    hide_insignificant=True,
    null_quantile=0.999,
)


kwargs = {
    "loc": "lower left",
    "ncols": 2,
    "columnspacing": 0.25,
    "handletextpad": 0.25,
    "reverse": True,
}
for ax in axes.flat:
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(left=1, right=1e3)
    ax.set_ylim(bottom=1e-7, top=1.5e-2)
    ax.legend(title="mouse", **kwargs)

fig.suptitle("mouse V1 single neuron responses to 2,800 natural images")

axes[0].set_xlabel("rank")
axes[0].set_ylabel("covariance")
axes[0].set_title("within-mouse", pad=12)

axes[1].set_xlabel("rank")
axes[1].set_ylabel("cross-covariance")
axes[1].set_title("cross-mouse,\nrelative to mouse 1")

fig.tight_layout()
save_figure(fig, filepath=FIGURES_HOME / "mouse.pdf")